In [ ]:
'''

IMPORT MODEL ARCHITECTURE and DATA GENERATOR

'''

In [ ]:
print("dual")
MODEL_PATH = "dual_final.pth"   # trained model weights file

device = "cuda" if torch.cuda.is_available() else "cpu"

model = DualSwinFusionClassifier(dim=160, depth=12, heads=5, ws=8, drop=0.1)

state = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(state)
model = model.to(device)
model.eval()

In [ ]:
print(f"Loaded model from: {MODEL_PATH}")

# --------------------------------------------
# 3. Get probabilities on validation loader
# --------------------------------------------
probs_dualtr, y_val_dualtr = predict_probs_cached(
    model,
    val_loader,
    device=device,
    use_amp=True,
    desc="validation inference"
)

print("probs_val shape:", probs_val.shape)
print("y_val shape:", y_val.shape)
print("First 10 probs:", probs_val[:10])

In [ ]:
THRESHOLD = 0.5
UA_coef = 0.372   # UA of model
PA_coef = 0.746

# probs_val must already exist and correspond to val_loader / val_idx order
pred_labels = (np.asarray(probs_dualtr).reshape(-1) >= THRESHOLD).astype(int)

# ---------------------------------------------------
# build per-image table for validation set
# ---------------------------------------------------
rows = []
for k, sample_idx in enumerate(val_idx):
    s = samples[sample_idx]
    meta = parse_area_and_true_from_filename(s.path)
    meta["pred"] = int(pred_labels[k])
    rows.append(meta)

df_area = pd.DataFrame(rows)

# ---------------------------------------------------
# area sums
# ---------------------------------------------------
true_area_sum = df_area.loc[df_area["true"] == 1, "area"].sum()
pred_area_sum = df_area.loc[df_area["pred"] == 1, "area"].sum()
pred_area_sum_ua = pred_area_sum * UA_coef/PA_coef

print(f"UA_coef = {UA_coef}, PA_coef = {PA_coef}")
print(f"Area sum by true class=1: {true_area_sum:.6f}")
print(f"Area sum by predicted class=1: {pred_area_sum:.6f}")
print(f"Area sum by predicted class=1 * UA_coef: {pred_area_sum_ua:.6f}")